In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
print("\n" + "="*60)
print("DATA LOADING & PREPROCESSING")
print("="*60)

train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")
tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Unique tags: {len(tag_vocab)}")

label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)
train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
test_data["EncodedTags"] = label_encoder.transform(test_data["language"])
num_classes = len(tag_vocab)

print(f"\nLabel distribution in training data:")
print(train_data["language"].value_counts())


In [ ]:

print("\n" + "="*60)
print("INITIALIZING GraphCodeBERT TOKENIZER")
print("="*60)

model_name = "microsoft/graphcodebert-base"
print(f"Loading GraphCodeBERT model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer class: {type(tokenizer).__name__}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: CLS={tokenizer.cls_token}, SEP={tokenizer.sep_token}, PAD={tokenizer.pad_token}")

def tokenize_code(code, max_length=512):
    """
    Tokenize code for GraphCodeBERT.
    Note: Full GraphCodeBERT would require data flow edges, but we'll start with text only.
    """
    return tokenizer(
        code,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

sample_code = "def hello_world():\n    print('Hello, World!')"
sample_tokens = tokenize_code(sample_code)
print(f"\nSample tokenization test:")
print(f"  Input shape: {sample_tokens['input_ids'].shape}")
print(f"  Attention mask shape: {sample_tokens['attention_mask'].shape}")
print(f"  First 10 tokens: {tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'][0][:10])}")


In [ ]:
class CodeDataset(Dataset):
    def __init__(self, dataframe, max_length=512):
        self.df = dataframe
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = tokenize_code(row["code"], self.max_length)
        
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedTags"], dtype=torch.long)
        }

max_seq_length = 512
print(f"\nUsing sequence length: {max_seq_length}")

train_dataset = CodeDataset(train_data, max_seq_length)
test_dataset = CodeDataset(test_data, max_seq_length)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")


In [ ]:
print("\n" + "="*60)
print("STAGE 1 MODEL: GraphCodeBERT Basic Classifier")
print("="*60)

class GraphCodeBERTStage1(nn.Module):
    """
    Stage 1: GraphCodeBERT for classification.
    GraphCodeBERT is based on RoBERTa, so we can use mean pooling like CodeBERT.
    """
    def __init__(self, num_classes, model_name="microsoft/graphcodebert-base", use_mean_pooling=True):
        super().__init__()
        
        print(f"Loading GraphCodeBERT model: {model_name}")
        
        self.graphcodebert = AutoModel.from_pretrained(model_name)
        
        self.config = self.graphcodebert.config
        hidden_size = self.config.hidden_size
        
        print(f"GraphCodeBERT Configuration:")
        print(f"  - Hidden size: {hidden_size}")
        print(f"  - Number of layers: {self.config.num_hidden_layers}")
        print(f"  - Number of attention heads: {self.config.num_attention_heads}")
        print(f"  - Intermediate size: {self.config.intermediate_size}")
        print(f"  - Model type: {self.config.model_type}")
        print(f"  - Vocab size: {self.config.vocab_size}")
        
        self.use_mean_pooling = use_mean_pooling
        
        self.classifier = nn.Linear(hidden_size, num_classes)
        
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)
        
        nn.init.xavier_uniform_(self.classifier.weight)
        if self.classifier.bias is not None:
            nn.init.zeros_(self.classifier.bias)
        
        print(f"  Using {'mean pooling' if use_mean_pooling else 'CLS token'} for classification")
        
    def forward(self, input_ids, attention_mask):
        outputs = self.graphcodebert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        last_hidden_state = outputs.last_hidden_state
        
        if self.use_mean_pooling:
            attention_mask_expanded = attention_mask.unsqueeze(-1).float()
            weighted_sum = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
            token_count = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
            pooled_output = weighted_sum / token_count
        else:
            pooled_output = last_hidden_state[:, 0, :]
        
        pooled_output = self.dropout(pooled_output)
        
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
print("\n" + "="*60)
print("STAGE 2 MODEL: GraphCodeBERT Layer Representation Transformer")
print("="*60)

class GraphCodeBERT_RI_Transformer(nn.Module):
    """
    Stage 2: GraphCodeBERT with layer representation information extraction.
    This follows your exact CodeBERT approach but with GraphCodeBERT as base.
    """
    def __init__(self, num_classes, model_name="microsoft/graphcodebert-base", use_mean_pooling=True):
        super().__init__()
        
        print(f"Loading GraphCodeBERT model with hidden states: {model_name}")
        
        self.graphcodebert = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True
        )
        
        self.config = self.graphcodebert.config
        hidden_size = self.config.hidden_size
        num_layers = self.config.num_hidden_layers
        
        print(f"\nGraphCodeBERT RI Transformer Configuration:")
        print(f"  - Hidden size: {hidden_size}")
        print(f"  - Number of layers: {num_layers}")
        print(f"  - Model type: {self.config.model_type}")
        
        self.use_mean_pooling = use_mean_pooling
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=self.config.num_attention_heads,
            dim_feedforward=self.config.intermediate_size,
            dropout=self.config.hidden_dropout_prob,
            batch_first=True,
            activation='gelu'
        )
        
        self.layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        
        self.attention_fc = nn.Linear(hidden_size, hidden_size)
        self.context_vector = nn.Parameter(torch.randn(hidden_size))
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(self.config.hidden_dropout_prob),
            nn.Linear(hidden_size, num_classes)
        )
        
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)
        
        self._init_weights()
        
    def _init_weights(self):
        """Initialize weights for newly added layers"""
        for module in [self.layer_transformer, self.attention_fc, self.classifier]:
            if hasattr(module, 'weight'):
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
        
        nn.init.xavier_uniform_(self.context_vector.unsqueeze(0))
        
    def forward(self, input_ids, attention_mask):
        outputs = self.graphcodebert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        
      
        hidden_states = outputs.hidden_states
        
      
        transformer_hidden_states = hidden_states[1:]
        
        batch_size = input_ids.size(0)
        layerwise_embeddings = []
        
        for layer_hidden in transformer_hidden_states:

            if self.use_mean_pooling:
                attention_mask_expanded = attention_mask.unsqueeze(-1).float()
                weighted_sum = torch.sum(layer_hidden * attention_mask_expanded, dim=1)
                token_count = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
                layer_representation = weighted_sum / token_count
            else:
                layer_representation = layer_hidden[:, 0, :]
            
            layer_representation = self.dropout(layer_representation)
            layerwise_embeddings.append(layer_representation)
        
        layer_sequence = torch.stack(layerwise_embeddings, dim=1)
        
        layer_sequence = nn.LayerNorm(layer_sequence.size(-1)).to(layer_sequence.device)(layer_sequence)
        
        transformed_layers = self.layer_transformer(layer_sequence)
        
        u = torch.tanh(self.attention_fc(transformed_layers))
        
        scores = torch.matmul(u, self.context_vector)  
        
        alpha = torch.softmax(scores, dim=1)
        
        weighted_output = torch.sum(transformed_layers * alpha.unsqueeze(-1), dim=1)
        
        weighted_output = self.dropout(weighted_output)
        logits = self.classifier(weighted_output)
        
        return logits, alpha

In [ ]:
def count_parameters(model):
    """Count trainable and total parameters"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def evaluate_model(model, data_loader, device, is_stage2=False):
    """Evaluate model on given data loader"""
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            if is_stage2:
                logits, _ = model(input_ids, attention_mask)
            else:
                logits = model(input_ids, attention_mask)
            
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            all_predictions.extend(predictions)
            all_labels.extend(labels)
    
    accuracy = accuracy_score(all_labels, all_predictions)
    return accuracy, all_predictions, all_labels

In [ ]:
print("\n" + "="*60)
print("STAGE 1 TRAINING: GraphCodeBERT Basic Classifier")
print("="*60)

stage1_model = GraphCodeBERTStage1(num_classes, model_name, use_mean_pooling=True).to(device)

total_params, trainable_params = count_parameters(stage1_model)
print(f"\nStage 1 Model Parameters:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable percentage: {trainable_params/total_params*100:.2f}%")

optimizer_stage1 = optim.AdamW(
    stage1_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)
criterion = nn.CrossEntropyLoss()

epochs_stage1 = 1
stage1_model.train()

for epoch in range(epochs_stage1):
    total_loss = 0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader, 1):
        optimizer_stage1.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = stage1_model(input_ids, attention_mask)
        
        loss = criterion(logits, labels)
        
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage1_model.parameters(), max_norm=1.0)
        optimizer_stage1.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            accuracy = correct / total
            print(f"  Stage1 - Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, "
                  f"Acc: {accuracy:.4f}")
    
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"\n[Stage 1] Epoch {epoch+1} Complete")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Training Accuracy: {epoch_accuracy:.4f}")
    
    val_accuracy, _, _ = evaluate_model(stage1_model, test_loader, device, is_stage2=False)
    print(f"  Validation Accuracy: {val_accuracy:.4f}")

print("\nSaving Stage 1 GraphCodeBERT weights...")
torch.save(
    {
        'model_state_dict': stage1_model.graphcodebert.state_dict(),
        'config': stage1_model.config.to_dict(),
        'classifier_state_dict': stage1_model.classifier.state_dict(),
        'use_mean_pooling': stage1_model.use_mean_pooling
    },
    "/home/aman_swaraj/Downloads/Codelite/graphcodebert_stage1_weights.pt"
)
print("Stage 1 weights saved successfully.")

In [ ]:
print("\n" + "="*60)
print("STAGE 2 TRAINING: GraphCodeBERT Layer Representation Transformer")
print("="*60)

stage2_model = GraphCodeBERT_RI_Transformer(num_classes, model_name, use_mean_pooling=True).to(device)

print("Loading Stage 1 weights into Stage 2 model...")
try:
    checkpoint = torch.load("/home/aman_swaraj/Downloads/Codelite/graphcodebert_stage1_weights.pt")
    stage2_model.graphcodebert.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Stage 1 GraphCodeBERT weights loaded successfully")
except Exception as e:
    print(f"✗ Error loading weights: {e}")
    print("Initializing with fresh pretrained weights...")

for name, param in stage2_model.graphcodebert.named_parameters():
    param.requires_grad = False

total_params_stage2, trainable_params_stage2 = count_parameters(stage2_model)
frozen_params_stage2 = total_params_stage2 - trainable_params_stage2

print(f"\nStage 2 Parameter Summary:")
print(f"  Total parameters: {total_params_stage2:,}")
print(f"  Trainable parameters: {trainable_params_stage2:,}")
print(f"  Frozen parameters: {frozen_params_stage2:,}")
print(f"  Trainable percentage: {trainable_params_stage2/total_params_stage2*100:.2f}%")

optimizer_stage2 = optim.AdamW(
    filter(lambda p: p.requires_grad, stage2_model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

epochs_stage2 = 2
stage2_model.train()

for epoch in range(epochs_stage2):
    total_loss = 0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader, 1):
        optimizer_stage2.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits, attention_weights = stage2_model(input_ids, attention_mask)
        
        loss = criterion(logits, labels)
        
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, stage2_model.parameters()),
            max_norm=1.0
        )
        optimizer_stage2.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            accuracy = correct / total
            avg_attention = attention_weights.mean(dim=0).cpu().detach().numpy()
            dominant_layer = avg_attention.argmax()
            
            print(f"  Stage2 - Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, "
                  f"Acc: {accuracy:.4f}, Dom Layer: {dominant_layer}")
    
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"\n[Stage 2] Epoch {epoch+1} Complete")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Training Accuracy: {epoch_accuracy:.4f}")
    
    val_accuracy, _, _ = evaluate_model(stage2_model, test_loader, device, is_stage2=True)
    print(f"  Validation Accuracy: {val_accuracy:.4f}")

print("\nSaving complete Stage 2 model...")
torch.save(
    {
        'model_state_dict': stage2_model.state_dict(),
        'config': stage2_model.config.to_dict(),
        'num_classes': num_classes,
        'label_encoder_classes': label_encoder.classes_.tolist(),
        'use_mean_pooling': stage2_model.use_mean_pooling
    },
    "/home/aman_swaraj/Downloads/Codelite/graphcodebert_full_RI_transformer.pt"
)
print("Stage 2 model saved successfully.")

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE EVALUATION ON TEST SET")
print("="*60)

import numpy as np
from scipy import stats
from sklearn.metrics import precision_recall_fscore_support

def get_model_predictions(model, data_loader, device, is_stage2=False):
    """Get predictions from model with optional attention weights for Stage 2"""
    model.eval()
    all_predictions = []
    all_probabilities = []
    all_true_labels = []
    all_attention_weights = [] if is_stage2 else None
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            if is_stage2:
                logits, attention_weights = model(input_ids, attention_mask)
                all_attention_weights.append(attention_weights.cpu().numpy())
            else:
                logits = model(input_ids, attention_mask)
            
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_predictions.extend(predictions)
            all_probabilities.append(probabilities)
            all_true_labels.extend(labels)
    
    all_probabilities = np.concatenate(all_probabilities, axis=0)
    
    if is_stage2 and all_attention_weights:
        all_attention_weights = np.concatenate(all_attention_weights, axis=0)
    
    return {
        'predictions': np.array(all_predictions),
        'probabilities': all_probabilities,
        'true_labels': np.array(all_true_labels),
        'attention_weights': all_attention_weights if is_stage2 else None
    }

print("\n1. EVALUATING STAGE 1 MODEL (Basic GraphCodeBERT Classifier)")
print("-" * 60)

stage1_results = get_model_predictions(stage1_model, test_loader, device, is_stage2=False)

stage1_accuracy = accuracy_score(stage1_results['true_labels'], stage1_results['predictions'])
stage1_precision, stage1_recall, stage1_f1, _ = precision_recall_fscore_support(
    stage1_results['true_labels'], 
    stage1_results['predictions'],
    average='weighted'
)

print(f"Stage 1 Model Results:")
print(f"  Accuracy:  {stage1_accuracy:.4f}")
print(f"  Precision: {stage1_precision:.4f}")
print(f"  Recall:    {stage1_recall:.4f}")
print(f"  F1-Score:  {stage1_f1:.4f}")

print("\n2. EVALUATING STAGE 2 MODEL (Layer RI Transformer)")
print("-" * 60)

stage2_results = get_model_predictions(stage2_model, test_loader, device, is_stage2=True)

stage2_accuracy = accuracy_score(stage2_results['true_labels'], stage2_results['predictions'])
stage2_precision, stage2_recall, stage2_f1, _ = precision_recall_fscore_support(
    stage2_results['true_labels'], 
    stage2_results['predictions'],
    average='weighted'
)

print(f"Stage 2 Model Results:")
print(f"  Accuracy:  {stage2_accuracy:.4f}")
print(f"  Precision: {stage2_precision:.4f}")
print(f"  Recall:    {stage2_recall:.4f}")
print(f"  F1-Score:  {stage2_f1:.4f}")

print("\n3. STATISTICAL SIGNIFICANCE ANALYSIS")
print("-" * 60)

contingency_table = np.zeros((2, 2))
for i in range(len(stage1_results['true_labels'])):
    true_label = stage1_results['true_labels'][i]
    stage1_correct = (stage1_results['predictions'][i] == true_label)
    stage2_correct = (stage2_results['predictions'][i] == true_label)
    
    if stage1_correct and stage2_correct:
        contingency_table[0, 0] += 1
    elif stage1_correct and not stage2_correct:
        contingency_table[0, 1] += 1
    elif not stage1_correct and stage2_correct:
        contingency_table[1, 0] += 1
    else:
        contingency_table[1, 1] += 1

b = contingency_table[0, 1]
c = contingency_table[1, 0]
if b + c > 0:
    chi2_stat = ((abs(b - c) - 1)**2) / (b + c)  
    p_value = 1 - stats.chi2.cdf(chi2_stat, df=1)
else:
    chi2_stat = 0
    p_value = 1.0

print(f"McNemar's Test:")
print(f"  Contingency table: Both correct={contingency_table[0, 0]}, "
      f"Only Stage1={contingency_table[0, 1]}, Only Stage2={contingency_table[1, 0]}, Both wrong={contingency_table[1, 1]}")
print(f"  Chi-squared: {chi2_stat:.4f}, p-value: {p_value:.6f}")
print(f"  Significant at α=0.05? {'YES' if p_value < 0.05 else 'NO'}")

print("\n4. LAYER ATTENTION ANALYSIS (Stage 2)")
print("-" * 60)

if stage2_results['attention_weights'] is not None:
    attention_weights = stage2_results['attention_weights']
    num_layers = attention_weights.shape[1]
    
    print(f"Number of GraphCodeBERT layers analyzed: {num_layers}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    avg_attention_per_layer = attention_weights.mean(axis=0)
    std_attention_per_layer = attention_weights.std(axis=0)
    
    print("\nAverage Attention per GraphCodeBERT Layer:")
    print("-" * 50)
    print(f"{'Layer':<10} {'Mean Attention':<15} {'Std Dev':<15} {'Importance':<15}")
    print("-" * 70)
    
    for layer_idx in range(num_layers):
        mean_att = avg_attention_per_layer[layer_idx]
        std_att = std_attention_per_layer[layer_idx]
        
        if mean_att > avg_attention_per_layer.mean() + avg_attention_per_layer.std():
            importance = "VERY HIGH"
        elif mean_att > avg_attention_per_layer.mean():
            importance = "HIGH"
        elif mean_att > avg_attention_per_layer.mean() - avg_attention_per_layer.std():
            importance = "MEDIUM"
        else:
            importance = "LOW"
        
        print(f"Layer {layer_idx+1:<3} {mean_att:<15.4f} {std_att:<15.4f} {importance:<15}")
    
    top_layers = np.argsort(avg_attention_per_layer)[-3:][::-1]  # Top 3 layers
    print(f"\nTop 3 Most Important Layers: {[l+1 for l in top_layers]}")

print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

results_df = pd.DataFrame({
    "Code": test_data["code"].values,
    "True_Label": label_encoder.inverse_transform(stage1_results['true_labels']),
    "Stage1_Prediction": label_encoder.inverse_transform(stage1_results['predictions']),
    "Stage1_Correct": (stage1_results['predictions'] == stage1_results['true_labels']).astype(int),
    "Stage2_Prediction": label_encoder.inverse_transform(stage2_results['predictions']),
    "Stage2_Correct": (stage2_results['predictions'] == stage2_results['true_labels']).astype(int),
    "Agreement": (stage1_results['predictions'] == stage2_results['predictions']).astype(int)
})

if stage2_results['attention_weights'] is not None:
    attention_weights = stage2_results['attention_weights']
    num_layers = attention_weights.shape[1]
    
    for layer_idx in range(num_layers):
        results_df[f"Layer_{layer_idx+1}_Attention"] = attention_weights[:, layer_idx]

results_path = "/home/aman_swaraj/Downloads/Codelite/graphcodebert_RI_transformer_results.csv"
results_df.to_csv(results_path, index=False)

print(f"\nResults saved to: {results_path}")
print(f"Total test samples: {len(results_df)}")
print(f"Stage 1 Accuracy: {results_df['Stage1_Correct'].sum() / len(results_df):.4f}")
print(f"Stage 2 Accuracy: {results_df['Stage2_Correct'].sum() / len(results_df):.4f}")
print(f"Models agree on: {results_df['Agreement'].sum() / len(results_df):.2%} of samples")
print(f"Improvement: {(stage2_accuracy/stage1_accuracy - 1)*100:+.2f}%")
